In [ ]:
import os
# ============================================================
# RELIABILITY AND AGREEMENT ANALYSIS
# Original uninjected dataset, seed 42
# ============================================================
#
# This notebook generates results for:
#
# \subsection{Reliability and Agreement Analysis}
#     \subsubsection{Agreement-versus-Score Analysis}
#     \subsubsection{Comparison of the Pure and Full Variants}
#
# Main operations:
#   1. Load the current internal anomaly labels.
#   2. Reconstruct Full internal agreement across 9 signals.
#   3. Reconstruct Pure internal agreement across 3 signals.
#   4. Load saved Full and Pure reliability arrays.
#   5. Load pre-final-clipping fused scores generated by
#      04_Baseline_Comparison_Clean.ipynb.
#   6. Analyze score, reliability, and final selection by
#      agreement level.
#   7. Compare Full and Pure using normalized consensus.
#   8. Analyze shared, Full-only, and Pure-only detections.
#   9. Save Excel, CSV, LaTeX tables, NumPy arrays, and figures.
#
# No model training or inference is performed.
# No existing files are overwritten outside the new output folder.
# No Python variables are deleted.
# ============================================================


from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from openpyxl.styles import Font, Alignment


# ============================================================
# 1. Folder structure
# ============================================================

ROOT_DIR = Path(
    os.environ.get("GIS_ANOMALY_WORKSPACE_ROOT", ".")
    r"\Outlier Paper\Code"
)

PROPOSED_DIR = (
    ROOT_DIR
    / "03_Proposed_Results_Seed_42"
)

BASELINE_COMPARISON_DIR = (
    ROOT_DIR
    / "04_Baseline_Comparison_Results_Seed_42"
)

OUTPUT_DIR = (
    ROOT_DIR
    / "05_Reliability_Agreement_Results_Seed_42"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Input files
# ============================================================

RELIABILITY_PATH = (
    PROPOSED_DIR
    / "full_pure_reliability_diagnostics.npz"
)


# Exact top-2% masks generated by
# 04_Baseline_Comparison_Clean.ipynb
FULL_SELECTION_PATH = (
    BASELINE_COMPARISON_DIR
    / "Full_top2pct_anomaly_mask.npy"
)

PURE_SELECTION_PATH = (
    BASELINE_COMPARISON_DIR
    / "Pure_top2pct_anomaly_mask.npy"
)


# Reconstructed final fused scores before final clipping
FULL_SCORE_PATH = (
    BASELINE_COMPARISON_DIR
    / "Full_final_fused_score_preclip.npy"
)

PURE_SCORE_PATH = (
    BASELINE_COMPARISON_DIR
    / "Pure_final_fused_score_preclip.npy"
)


# Previously stored masks, used only for auditing
OLD_FULL_SELECTION_PATH = (
    PROPOSED_DIR
    / "full_final_anomaly_labels.npy"
)

OLD_PURE_SELECTION_PATH = (
    PROPOSED_DIR
    / "pure_final_anomaly_labels.npy"
)


# ============================================================
# 3. Output files
# ============================================================

OUTPUT_EXCEL = (
    OUTPUT_DIR
    / "Reliability_Agreement_Analysis_Results.xlsx"
)

FULL_AGREEMENT_PATH = (
    OUTPUT_DIR
    / "Full_internal_agreement_count.npy"
)

PURE_AGREEMENT_PATH = (
    OUTPUT_DIR
    / "Pure_internal_agreement_count.npy"
)

FULL_CONSENSUS_PATH = (
    OUTPUT_DIR
    / "Full_normalized_consensus.npy"
)

PURE_CONSENSUS_PATH = (
    OUTPUT_DIR
    / "Pure_normalized_consensus.npy"
)

SELECTION_SUBSET_PATH = (
    OUTPUT_DIR
    / "Pure_Full_selection_subset_labels.npy"
)


# ============================================================
# 4. Analysis settings
# ============================================================

ANOMALY_BUDGET = 0.02

FULL_SIGNAL_COUNT = 9
PURE_SIGNAL_COUNT = 3

# Common high-consensus definition.
#
# Full: at least 6 of 9 signals.
# Pure: at least 2 of 3 signals.
HIGH_CONSENSUS_FRACTION = 2.0 / 3.0

FULL_HIGH_CONSENSUS_COUNT = int(
    np.ceil(
        HIGH_CONSENSUS_FRACTION
        * FULL_SIGNAL_COUNT
    )
)

PURE_HIGH_CONSENSUS_COUNT = int(
    np.ceil(
        HIGH_CONSENSUS_FRACTION
        * PURE_SIGNAL_COUNT
    )
)


# ============================================================
# 5. General validation functions
# ============================================================

def require_file(path):
    """Stop execution if a required input is unavailable."""

    if not path.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{path}"
        )


def load_numeric_vector(
    path,
    expected_length=None,
    name=None,
    mmap_mode="r",
):
    """Load and validate a one-dimensional numeric array."""

    require_file(path)

    array = np.load(
        path,
        mmap_mode=mmap_mode,
    )

    array = np.asarray(
        array
    ).reshape(-1)

    array_name = (
        name
        if name is not None
        else path.name
    )

    if expected_length is not None:
        if array.size != expected_length:
            raise ValueError(
                f"{array_name} contains "
                f"{array.size:,} values, but "
                f"{expected_length:,} were expected."
            )

    if not np.issubdtype(
        array.dtype,
        np.number,
    ):
        raise TypeError(
            f"{array_name} is not numeric."
        )

    if not np.isfinite(array).all():
        bad_count = int(
            (~np.isfinite(array)).sum()
        )

        raise ValueError(
            f"{array_name} contains "
            f"{bad_count:,} non-finite values."
        )

    return array


def validate_binary_array(
    array,
    expected_length=None,
    name="Binary array",
):
    """Validate and return a zero-one uint8 vector."""

    array = np.asarray(
        array
    ).reshape(-1)

    if expected_length is not None:
        if array.size != expected_length:
            raise ValueError(
                f"{name} contains {array.size:,} values, "
                f"but {expected_length:,} were expected."
            )

    if not np.isfinite(array).all():
        raise ValueError(
            f"{name} contains non-finite values."
        )

    unique_values = np.unique(
        array
    )

    if not np.all(
        np.isin(
            unique_values,
            [0, 1, False, True],
        )
    ):
        raise ValueError(
            f"{name} is not binary. "
            f"Observed values: {unique_values}"
        )

    return array.astype(
        np.uint8,
        copy=False,
    )


def load_binary_vector(
    path,
    expected_length=None,
    name=None,
):
    """Load a binary NumPy vector."""

    require_file(path)

    array = np.load(
        path,
        mmap_mode="r",
    )

    return validate_binary_array(
        array=array,
        expected_length=expected_length,
        name=(
            name
            if name is not None
            else path.name
        ),
    )


def jaccard_similarity(
    first_mask,
    second_mask,
):
    """Return intersection, union, and Jaccard similarity."""

    first_mask = np.asarray(
        first_mask,
        dtype=bool,
    )

    second_mask = np.asarray(
        second_mask,
        dtype=bool,
    )

    intersection = int(
        np.logical_and(
            first_mask,
            second_mask,
        ).sum()
    )

    union = int(
        np.logical_or(
            first_mask,
            second_mask,
        ).sum()
    )

    jaccard = (
        intersection / union
        if union > 0
        else np.nan
    )

    return (
        intersection,
        union,
        jaccard,
    )


def exact_pearson_correlation(
    first_values,
    second_values,
):
    """
    Calculate exact Pearson correlation without constructing
    a two-column correlation matrix.
    """

    x = np.asarray(
        first_values,
        dtype=np.float64,
    ).reshape(-1)

    y = np.asarray(
        second_values,
        dtype=np.float64,
    ).reshape(-1)

    if x.size != y.size:
        raise ValueError(
            "Correlation vectors have different lengths."
        )

    n = float(
        x.size
    )

    sum_x = float(
        np.sum(
            x,
            dtype=np.float64,
        )
    )

    sum_y = float(
        np.sum(
            y,
            dtype=np.float64,
        )
    )

    sum_x_squared = float(
        np.sum(
            x * x,
            dtype=np.float64,
        )
    )

    sum_y_squared = float(
        np.sum(
            y * y,
            dtype=np.float64,
        )
    )

    sum_xy = float(
        np.sum(
            x * y,
            dtype=np.float64,
        )
    )

    numerator = (
        n * sum_xy
        - sum_x * sum_y
    )

    first_denominator = (
        n * sum_x_squared
        - sum_x ** 2
    )

    second_denominator = (
        n * sum_y_squared
        - sum_y ** 2
    )

    denominator = np.sqrt(
        first_denominator
        * second_denominator
    )

    if denominator == 0:
        return np.nan

    return (
        numerator
        / denominator
    )


# ============================================================
# 6. Check the required files
# ============================================================

REQUIRED_FILES = [
    RELIABILITY_PATH,
    FULL_SELECTION_PATH,
    PURE_SELECTION_PATH,
    FULL_SCORE_PATH,
    PURE_SCORE_PATH,
]

for required_path in REQUIRED_FILES:
    require_file(required_path)

print("All required files were found.")




In [ ]:
# ============================================================
# 7. Load Full and Pure pre-final-clipping scores
# ============================================================

full_score = load_numeric_vector(
    FULL_SCORE_PATH,
    name="Full pre-final-clipping fused score",
)

n_samples = full_score.size

pure_score = load_numeric_vector(
    PURE_SCORE_PATH,
    expected_length=n_samples,
    name="Pure pre-final-clipping fused score",
)

print("\nDataset size:")
print(f"{n_samples:,} pixels")

print("\nFull score range:")
print(
    float(full_score.min()),
    "to",
    float(full_score.max()),
)

print("\nPure score range:")
print(
    float(pure_score.min()),
    "to",
    float(pure_score.max()),
)


# ============================================================
# 8. Load exact top-2% final selections
# ============================================================

full_selected = load_binary_vector(
    FULL_SELECTION_PATH,
    expected_length=n_samples,
    name="Full exact top-2% mask",
).astype(
    bool,
    copy=False,
)

pure_selected = load_binary_vector(
    PURE_SELECTION_PATH,
    expected_length=n_samples,
    name="Pure exact top-2% mask",
).astype(
    bool,
    copy=False,
)

expected_selected_count = int(
    np.ceil(
        ANOMALY_BUDGET
        * n_samples
    )
)

full_selected_count = int(
    full_selected.sum()
)

pure_selected_count = int(
    pure_selected.sum()
)

if full_selected_count != expected_selected_count:
    raise ValueError(
        "Full mask does not contain the expected exact "
        f"top-2% count. Expected {expected_selected_count:,}, "
        f"found {full_selected_count:,}."
    )

if pure_selected_count != expected_selected_count:
    raise ValueError(
        "Pure mask does not contain the expected exact "
        f"top-2% count. Expected {expected_selected_count:,}, "
        f"found {pure_selected_count:,}."
    )

print("\nExact final selections:")
print("Full:", f"{full_selected_count:,}")
print("Pure:", f"{pure_selected_count:,}")


# ============================================================
# 9. Load the exact saved agreement, uncertainty,
#    and reliability arrays
# ============================================================

with np.load(
    RELIABILITY_PATH
) as reliability_file:

    print("\nKeys in full_pure_reliability_diagnostics.npz:")
    print(reliability_file.files)

    required_keys = [
        "full_agreement",
        "full_uncertainty",
        "full_reliability",
        "pure_agreement",
        "pure_uncertainty",
        "pure_reliability",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in reliability_file.files
    ]

    if missing_keys:
        raise KeyError(
            "The reliability archive is missing these keys: "
            + ", ".join(missing_keys)
            + "\nAvailable keys: "
            + ", ".join(reliability_file.files)
        )

    full_agreement_raw = np.asarray(
        reliability_file[
            "full_agreement"
        ]
    ).reshape(-1)

    full_uncertainty = np.asarray(
        reliability_file[
            "full_uncertainty"
        ]
    ).reshape(-1)

    full_reliability = np.asarray(
        reliability_file[
            "full_reliability"
        ]
    ).reshape(-1)

    pure_agreement_raw = np.asarray(
        reliability_file[
            "pure_agreement"
        ]
    ).reshape(-1)

    pure_uncertainty = np.asarray(
        reliability_file[
            "pure_uncertainty"
        ]
    ).reshape(-1)

    pure_reliability = np.asarray(
        reliability_file[
            "pure_reliability"
        ]
    ).reshape(-1)


# ============================================================
# 10. Validate saved diagnostic arrays
# ============================================================

diagnostic_arrays = {
    "Full agreement": full_agreement_raw,
    "Full uncertainty": full_uncertainty,
    "Full reliability": full_reliability,
    "Pure agreement": pure_agreement_raw,
    "Pure uncertainty": pure_uncertainty,
    "Pure reliability": pure_reliability,
}

for diagnostic_name, diagnostic_array in diagnostic_arrays.items():

    if diagnostic_array.size != n_samples:
        raise ValueError(
            f"{diagnostic_name} contains "
            f"{diagnostic_array.size:,} values, but "
            f"{n_samples:,} were expected."
        )

    if not np.isfinite(
        diagnostic_array
    ).all():
        raise ValueError(
            f"{diagnostic_name} contains non-finite values."
        )


# Agreement arrays must contain integer counts.
if not np.allclose(
    full_agreement_raw,
    np.round(full_agreement_raw),
):
    raise ValueError(
        "Full agreement contains non-integer values."
    )

if not np.allclose(
    pure_agreement_raw,
    np.round(pure_agreement_raw),
):
    raise ValueError(
        "Pure agreement contains non-integer values."
    )


full_agreement_count = np.round(
    full_agreement_raw
).astype(
    np.uint8,
    copy=False,
)

pure_agreement_count = np.round(
    pure_agreement_raw
).astype(
    np.uint8,
    copy=False,
)


if int(full_agreement_count.min()) < 0:
    raise ValueError(
        "Full agreement contains a value below zero."
    )

if int(full_agreement_count.max()) > FULL_SIGNAL_COUNT:
    raise ValueError(
        "Full agreement exceeds the expected maximum "
        f"of {FULL_SIGNAL_COUNT}."
    )

if int(pure_agreement_count.min()) < 0:
    raise ValueError(
        "Pure agreement contains a value below zero."
    )

if int(pure_agreement_count.max()) > PURE_SIGNAL_COUNT:
    raise ValueError(
        "Pure agreement exceeds the expected maximum "
        f"of {PURE_SIGNAL_COUNT}."
    )


reliability_tolerance = 1e-6

if (
    float(full_reliability.min())
    < -reliability_tolerance
    or float(full_reliability.max())
    > 1.0 + reliability_tolerance
):
    raise ValueError(
        "Full reliability lies outside the expected [0,1] range."
    )

if (
    float(pure_reliability.min())
    < -reliability_tolerance
    or float(pure_reliability.max())
    > 1.0 + reliability_tolerance
):
    raise ValueError(
        "Pure reliability lies outside the expected [0,1] range."
    )

if (
    float(full_uncertainty.min())
    < -reliability_tolerance
    or float(full_uncertainty.max())
    > 1.0 + reliability_tolerance
):
    raise ValueError(
        "Full uncertainty lies outside the expected [0,1] range."
    )

if (
    float(pure_uncertainty.min())
    < -reliability_tolerance
    or float(pure_uncertainty.max())
    > 1.0 + reliability_tolerance
):
    raise ValueError(
        "Pure uncertainty lies outside the expected [0,1] range."
    )


print("\nSaved diagnostics loaded successfully.")

print("\nObserved Full agreement range:")
print(
    int(full_agreement_count.min()),
    "to",
    int(full_agreement_count.max()),
)

print("\nObserved Pure agreement range:")
print(
    int(pure_agreement_count.min()),
    "to",
    int(pure_agreement_count.max()),
)

print("\nFull reliability range:")
print(
    float(full_reliability.min()),
    "to",
    float(full_reliability.max()),
)

print("\nPure reliability range:")
print(
    float(pure_reliability.min()),
    "to",
    float(pure_reliability.max()),
)


# ============================================================
# 11. Calculate normalized consensus
# ============================================================

full_normalized_consensus = (
    full_agreement_count.astype(
        np.float32
    )
    / float(
        FULL_SIGNAL_COUNT
    )
)

pure_normalized_consensus = (
    pure_agreement_count.astype(
        np.float32
    )
    / float(
        PURE_SIGNAL_COUNT
    )
)


# ============================================================
# 12. Save convenient copies in the new result folder
# ============================================================

np.save(
    FULL_AGREEMENT_PATH,
    full_agreement_count,
)

np.save(
    PURE_AGREEMENT_PATH,
    pure_agreement_count,
)

np.save(
    FULL_CONSENSUS_PATH,
    full_normalized_consensus,
)

np.save(
    PURE_CONSENSUS_PATH,
    pure_normalized_consensus,
)

np.save(
    OUTPUT_DIR
    / "Full_normalized_uncertainty.npy",
    full_uncertainty,
)

np.save(
    OUTPUT_DIR
    / "Pure_normalized_uncertainty.npy",
    pure_uncertainty,
)


# ============================================================
# 13. Agreement source audit
# ============================================================

agreement_audit_df = pd.DataFrame(
    [
        {
            "Variant": "Full",
            "Agreement source": (
                "Saved full_agreement array from "
                "full_pure_reliability_diagnostics.npz"
            ),
            "Internal signals": FULL_SIGNAL_COUNT,
            "Observed minimum": int(
                full_agreement_count.min()
            ),
            "Observed maximum": int(
                full_agreement_count.max()
            ),
            "Expected minimum": 0,
            "Expected maximum": FULL_SIGNAL_COUNT,
            "Length valid": (
                full_agreement_count.size
                == n_samples
            ),
            "Range valid": (
                full_agreement_count.min() >= 0
                and full_agreement_count.max()
                <= FULL_SIGNAL_COUNT
            ),
        },
        {
            "Variant": "Pure",
            "Agreement source": (
                "Saved pure_agreement array from "
                "full_pure_reliability_diagnostics.npz"
            ),
            "Internal signals": PURE_SIGNAL_COUNT,
            "Observed minimum": int(
                pure_agreement_count.min()
            ),
            "Observed maximum": int(
                pure_agreement_count.max()
            ),
            "Expected minimum": 0,
            "Expected maximum": PURE_SIGNAL_COUNT,
            "Length valid": (
                pure_agreement_count.size
                == n_samples
            ),
            "Range valid": (
                pure_agreement_count.min() >= 0
                and pure_agreement_count.max()
                <= PURE_SIGNAL_COUNT
            ),
        },
    ]
)

print("\nAgreement source audit:")
print(agreement_audit_df)

In [ ]:
# ============================================================
# 14. Validate current masks against older stored masks
# ============================================================

mask_audit_rows = []


for (
    variant_name,
    current_mask,
    old_mask_path,
) in [
    (
        "Full",
        full_selected,
        OLD_FULL_SELECTION_PATH,
    ),
    (
        "Pure",
        pure_selected,
        OLD_PURE_SELECTION_PATH,
    ),
]:

    if old_mask_path.exists():

        old_mask = load_binary_vector(
            old_mask_path,
            expected_length=n_samples,
            name=f"Old {variant_name} final mask",
        ).astype(
            bool,
            copy=False,
        )

        intersection, union, jaccard = (
            jaccard_similarity(
                current_mask,
                old_mask,
            )
        )

        mask_audit_rows.append(
            {
                "Variant": variant_name,
                "Old mask available": True,
                "Current selected pixels": int(
                    current_mask.sum()
                ),
                "Old selected pixels": int(
                    old_mask.sum()
                ),
                "Intersection": intersection,
                "Union": union,
                "Jaccard": jaccard,
                "Exactly identical": bool(
                    np.array_equal(
                        current_mask,
                        old_mask,
                    )
                ),
            }
        )

    else:

        mask_audit_rows.append(
            {
                "Variant": variant_name,
                "Old mask available": False,
                "Current selected pixels": int(
                    current_mask.sum()
                ),
                "Old selected pixels": np.nan,
                "Intersection": np.nan,
                "Union": np.nan,
                "Jaccard": np.nan,
                "Exactly identical": np.nan,
            }
        )


mask_audit_df = pd.DataFrame(
    mask_audit_rows
)


# ============================================================
# 16. Agreement-level summary function
# ============================================================

def create_agreement_level_summary(
    variant_name,
    agreement_count,
    total_signal_count,
    final_score,
    reliability,
    selected_mask,
):
    """
    Summarize score, reliability, and final selection at each
    exact internal agreement level.
    """

    rows = []

    total_selected = int(
        selected_mask.sum()
    )

    for agreement_level in range(
        total_signal_count + 1
    ):

        level_mask = (
            agreement_count
            == agreement_level
        )

        pixel_count = int(
            level_mask.sum()
        )

        normalized_consensus = (
            agreement_level
            / total_signal_count
        )

        if pixel_count == 0:

            rows.append(
                {
                    "Variant": variant_name,
                    "Agreement count": agreement_level,
                    "Total signals": total_signal_count,
                    "Normalized consensus": (
                        normalized_consensus
                    ),
                    "Pixel count": 0,
                    "Dataset share (%)": 0.0,
                    "Mean score": np.nan,
                    "Score Q1": np.nan,
                    "Median score": np.nan,
                    "Score Q3": np.nan,
                    "Mean reliability": np.nan,
                    "Reliability Q1": np.nan,
                    "Median reliability": np.nan,
                    "Reliability Q3": np.nan,
                    "Selected pixels": 0,
                    "Selection rate within level (%)": (
                        np.nan
                    ),
                    "Share of final selection (%)": 0.0,
                }
            )

            continue

        level_scores = final_score[
            level_mask
        ]

        level_reliability = reliability[
            level_mask
        ]

        selected_at_level = int(
            np.logical_and(
                level_mask,
                selected_mask,
            ).sum()
        )

        score_quartiles = np.percentile(
            level_scores,
            [
                25,
                50,
                75,
            ],
        )

        reliability_quartiles = np.percentile(
            level_reliability,
            [
                25,
                50,
                75,
            ],
        )

        rows.append(
            {
                "Variant": variant_name,
                "Agreement count": agreement_level,
                "Total signals": total_signal_count,
                "Normalized consensus": (
                    normalized_consensus
                ),
                "Pixel count": pixel_count,
                "Dataset share (%)": (
                    pixel_count
                    / n_samples
                    * 100.0
                ),
                "Mean score": float(
                    np.mean(
                        level_scores,
                        dtype=np.float64,
                    )
                ),
                "Score Q1": float(
                    score_quartiles[0]
                ),
                "Median score": float(
                    score_quartiles[1]
                ),
                "Score Q3": float(
                    score_quartiles[2]
                ),
                "Mean reliability": float(
                    np.mean(
                        level_reliability,
                        dtype=np.float64,
                    )
                ),
                "Reliability Q1": float(
                    reliability_quartiles[0]
                ),
                "Median reliability": float(
                    reliability_quartiles[1]
                ),
                "Reliability Q3": float(
                    reliability_quartiles[2]
                ),
                "Selected pixels": selected_at_level,
                "Selection rate within level (%)": (
                    selected_at_level
                    / pixel_count
                    * 100.0
                ),
                "Share of final selection (%)": (
                    selected_at_level
                    / total_selected
                    * 100.0
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


full_agreement_summary_df = (
    create_agreement_level_summary(
        variant_name="Full",
        agreement_count=full_agreement_count,
        total_signal_count=FULL_SIGNAL_COUNT,
        final_score=full_score,
        reliability=full_reliability,
        selected_mask=full_selected,
    )
)


pure_agreement_summary_df = (
    create_agreement_level_summary(
        variant_name="Pure",
        agreement_count=pure_agreement_count,
        total_signal_count=PURE_SIGNAL_COUNT,
        final_score=pure_score,
        reliability=pure_reliability,
        selected_mask=pure_selected,
    )
)


agreement_level_summary_df = pd.concat(
    [
        full_agreement_summary_df,
        pure_agreement_summary_df,
    ],
    ignore_index=True,
)


print("\nFull agreement-level summary:")
print(
    full_agreement_summary_df.round(6)
)

print("\nPure agreement-level summary:")
print(
    pure_agreement_summary_df.round(6)
)




In [ ]:
# ============================================================
# 17. Trend and correlation analysis
# ============================================================

def create_trend_statistics(
    variant_name,
    agreement_count,
    normalized_consensus,
    score,
    reliability,
    agreement_summary,
):
    """
    Calculate exact pixel-level Pearson correlations and
    agreement-level Spearman trend statistics.
    """

    nonempty_summary = agreement_summary[
        agreement_summary[
            "Pixel count"
        ] > 0
    ].copy()

    median_score_spearman = spearmanr(
        nonempty_summary[
            "Agreement count"
        ],
        nonempty_summary[
            "Median score"
        ],
    )

    median_reliability_spearman = spearmanr(
        nonempty_summary[
            "Agreement count"
        ],
        nonempty_summary[
            "Median reliability"
        ],
    )

    selection_rate_spearman = spearmanr(
        nonempty_summary[
            "Agreement count"
        ],
        nonempty_summary[
            "Selection rate within level (%)"
        ],
    )

    median_score_values = (
        nonempty_summary[
            "Median score"
        ]
        .to_numpy()
    )

    median_reliability_values = (
        nonempty_summary[
            "Median reliability"
        ]
        .to_numpy()
    )

    selection_rate_values = (
        nonempty_summary[
            "Selection rate within level (%)"
        ]
        .to_numpy()
    )

    score_nondecreasing_transitions = int(
        np.sum(
            np.diff(
                median_score_values
            )
            >= 0
        )
    )

    reliability_nondecreasing_transitions = int(
        np.sum(
            np.diff(
                median_reliability_values
            )
            >= 0
        )
    )

    selection_nondecreasing_transitions = int(
        np.sum(
            np.diff(
                selection_rate_values
            )
            >= 0
        )
    )

    possible_transitions = max(
        len(
            nonempty_summary
        )
        - 1,
        0,
    )

    return {
        "Variant": variant_name,
        "Pixel-level Pearson: consensus vs score": (
            exact_pearson_correlation(
                normalized_consensus,
                score,
            )
        ),
        "Pixel-level Pearson: consensus vs reliability": (
            exact_pearson_correlation(
                normalized_consensus,
                reliability,
            )
        ),
        "Pixel-level Pearson: reliability vs score": (
            exact_pearson_correlation(
                reliability,
                score,
            )
        ),
        "Agreement-level Spearman: median score": (
            float(
                median_score_spearman.statistic
            )
        ),
        "Agreement-level Spearman p-value: median score": (
            float(
                median_score_spearman.pvalue
            )
        ),
        "Agreement-level Spearman: median reliability": (
            float(
                median_reliability_spearman.statistic
            )
        ),
        "Agreement-level Spearman p-value: median reliability": (
            float(
                median_reliability_spearman.pvalue
            )
        ),
        "Agreement-level Spearman: selection rate": (
            float(
                selection_rate_spearman.statistic
            )
        ),
        "Agreement-level Spearman p-value: selection rate": (
            float(
                selection_rate_spearman.pvalue
            )
        ),
        "Nondecreasing median-score transitions": (
            score_nondecreasing_transitions
        ),
        "Nondecreasing median-reliability transitions": (
            reliability_nondecreasing_transitions
        ),
        "Nondecreasing selection-rate transitions": (
            selection_nondecreasing_transitions
        ),
        "Possible agreement-level transitions": (
            possible_transitions
        ),
    }


full_trend_statistics = create_trend_statistics(
    variant_name="Full",
    agreement_count=full_agreement_count,
    normalized_consensus=full_normalized_consensus,
    score=full_score,
    reliability=full_reliability,
    agreement_summary=full_agreement_summary_df,
)

pure_trend_statistics = create_trend_statistics(
    variant_name="Pure",
    agreement_count=pure_agreement_count,
    normalized_consensus=pure_normalized_consensus,
    score=pure_score,
    reliability=pure_reliability,
    agreement_summary=pure_agreement_summary_df,
)


trend_summary_df = pd.DataFrame(
    [
        full_trend_statistics,
        pure_trend_statistics,
    ]
)




In [ ]:
# ============================================================
# 18. Full-versus-Pure variant summary
# ============================================================

def summarize_variant(
    variant_name,
    signal_count,
    agreement_count,
    normalized_consensus,
    reliability,
    final_score,
    selected_mask,
    high_consensus_count,
):
    """Create one-row summary for Full or Pure."""

    selected_consensus = normalized_consensus[
        selected_mask
    ]

    selected_reliability = reliability[
        selected_mask
    ]

    selected_score = final_score[
        selected_mask
    ]

    high_consensus_all = (
        agreement_count
        >= high_consensus_count
    )

    high_consensus_selected = np.logical_and(
        high_consensus_all,
        selected_mask,
    )

    zero_agreement_selected = np.logical_and(
        agreement_count == 0,
        selected_mask,
    )

    selected_reliability_quartiles = (
        np.percentile(
            selected_reliability,
            [
                25,
                50,
                75,
            ],
        )
    )

    selected_consensus_quartiles = (
        np.percentile(
            selected_consensus,
            [
                25,
                50,
                75,
            ],
        )
    )

    return {
        "Variant": variant_name,
        "Internal signals": signal_count,
        "Selected pixels": int(
            selected_mask.sum()
        ),
        "Mean consensus, all pixels": float(
            np.mean(
                normalized_consensus,
                dtype=np.float64,
            )
        ),
        "Median consensus, all pixels": float(
            np.median(
                normalized_consensus
            )
        ),
        "Mean consensus, selected pixels": float(
            np.mean(
                selected_consensus,
                dtype=np.float64,
            )
        ),
        "Selected consensus Q1": float(
            selected_consensus_quartiles[0]
        ),
        "Median consensus, selected pixels": float(
            selected_consensus_quartiles[1]
        ),
        "Selected consensus Q3": float(
            selected_consensus_quartiles[2]
        ),
        "Mean reliability, all pixels": float(
            np.mean(
                reliability,
                dtype=np.float64,
            )
        ),
        "Median reliability, all pixels": float(
            np.median(
                reliability
            )
        ),
        "Mean reliability, selected pixels": float(
            np.mean(
                selected_reliability,
                dtype=np.float64,
            )
        ),
        "Selected reliability Q1": float(
            selected_reliability_quartiles[0]
        ),
        "Median reliability, selected pixels": float(
            selected_reliability_quartiles[1]
        ),
        "Selected reliability Q3": float(
            selected_reliability_quartiles[2]
        ),
        "Median pre-clipping score, selected pixels": float(
            np.median(
                selected_score
            )
        ),
        "High-consensus count threshold": (
            high_consensus_count
        ),
        "High-consensus share, all pixels (%)": (
            high_consensus_all.sum()
            / n_samples
            * 100.0
        ),
        "High-consensus share, selected pixels (%)": (
            high_consensus_selected.sum()
            / selected_mask.sum()
            * 100.0
        ),
        "Zero-agreement share, selected pixels (%)": (
            zero_agreement_selected.sum()
            / selected_mask.sum()
            * 100.0
        ),
        "Consensus lift: selected vs all": (
            np.mean(
                selected_consensus,
                dtype=np.float64,
            )
            / np.mean(
                normalized_consensus,
                dtype=np.float64,
            )
            if np.mean(
                normalized_consensus,
                dtype=np.float64,
            ) > 0
            else np.nan
        ),
        "Reliability lift: selected vs all": (
            np.mean(
                selected_reliability,
                dtype=np.float64,
            )
            / np.mean(
                reliability,
                dtype=np.float64,
            )
            if np.mean(
                reliability,
                dtype=np.float64,
            ) > 0
            else np.nan
        ),
    }


full_variant_summary = summarize_variant(
    variant_name="Full",
    signal_count=FULL_SIGNAL_COUNT,
    agreement_count=full_agreement_count,
    normalized_consensus=full_normalized_consensus,
    reliability=full_reliability,
    final_score=full_score,
    selected_mask=full_selected,
    high_consensus_count=(
        FULL_HIGH_CONSENSUS_COUNT
    ),
)

pure_variant_summary = summarize_variant(
    variant_name="Pure",
    signal_count=PURE_SIGNAL_COUNT,
    agreement_count=pure_agreement_count,
    normalized_consensus=pure_normalized_consensus,
    reliability=pure_reliability,
    final_score=pure_score,
    selected_mask=pure_selected,
    high_consensus_count=(
        PURE_HIGH_CONSENSUS_COUNT
    ),
)


variant_summary_df = pd.DataFrame(
    [
        full_variant_summary,
        pure_variant_summary,
    ]
)


print("\nFull-versus-Pure summary:")
print(
    variant_summary_df.T
)


# ============================================================
# 19. Full-Pure final-selection overlap
# ============================================================

both_selected = np.logical_and(
    full_selected,
    pure_selected,
)

full_only_selected = np.logical_and(
    full_selected,
    ~pure_selected,
)

pure_only_selected = np.logical_and(
    pure_selected,
    ~full_selected,
)

neither_selected = np.logical_and(
    ~full_selected,
    ~pure_selected,
)


selection_intersection = int(
    both_selected.sum()
)

selection_union = int(
    np.logical_or(
        full_selected,
        pure_selected,
    ).sum()
)

selection_jaccard = (
    selection_intersection
    / selection_union
)

selection_overlap_coefficient = (
    selection_intersection
    / min(
        full_selected_count,
        pure_selected_count,
    )
)


selection_overlap_df = pd.DataFrame(
    [
        {
            "Full selected pixels": (
                full_selected_count
            ),
            "Pure selected pixels": (
                pure_selected_count
            ),
            "Selected by both": (
                selection_intersection
            ),
            "Full-only selected": int(
                full_only_selected.sum()
            ),
            "Pure-only selected": int(
                pure_only_selected.sum()
            ),
            "Selection union": (
                selection_union
            ),
            "Jaccard similarity": (
                selection_jaccard
            ),
            "Overlap coefficient": (
                selection_overlap_coefficient
            ),
            "Shared share of Full selection (%)": (
                selection_intersection
                / full_selected_count
                * 100.0
            ),
            "Shared share of Pure selection (%)": (
                selection_intersection
                / pure_selected_count
                * 100.0
            ),
        }
    ]
)


# ============================================================
# 20. Label and save selection subsets
# ============================================================

# Codes:
#   0 = neither selected
#   1 = selected by both
#   2 = Full only
#   3 = Pure only

selection_subset_labels = np.zeros(
    n_samples,
    dtype=np.uint8,
)

selection_subset_labels[
    both_selected
] = 1

selection_subset_labels[
    full_only_selected
] = 2

selection_subset_labels[
    pure_only_selected
] = 3


np.save(
    SELECTION_SUBSET_PATH,
    selection_subset_labels,
)


# ============================================================
# 21. Subset-specific Full and Pure characteristics
# ============================================================

SELECTION_SUBSETS = {
    "Both selected": both_selected,
    "Full only": full_only_selected,
    "Pure only": pure_only_selected,
    "Neither selected": neither_selected,
}


subset_rows = []


for subset_name, subset_mask in (
    SELECTION_SUBSETS.items()
):

    subset_count = int(
        subset_mask.sum()
    )

    if subset_count == 0:

        subset_rows.append(
            {
                "Selection subset": subset_name,
                "Pixel count": 0,
                "Dataset share (%)": 0.0,
                "Mean Full consensus": np.nan,
                "Median Full consensus": np.nan,
                "Mean Pure consensus": np.nan,
                "Median Pure consensus": np.nan,
                "Mean Full reliability": np.nan,
                "Median Full reliability": np.nan,
                "Mean Pure reliability": np.nan,
                "Median Pure reliability": np.nan,
                "Median Full pre-clipping score": np.nan,
                "Median Pure pre-clipping score": np.nan,
            }
        )

        continue

    subset_rows.append(
        {
            "Selection subset": subset_name,
            "Pixel count": subset_count,
            "Dataset share (%)": (
                subset_count
                / n_samples
                * 100.0
            ),
            "Mean Full consensus": float(
                np.mean(
                    full_normalized_consensus[
                        subset_mask
                    ],
                    dtype=np.float64,
                )
            ),
            "Median Full consensus": float(
                np.median(
                    full_normalized_consensus[
                        subset_mask
                    ]
                )
            ),
            "Mean Pure consensus": float(
                np.mean(
                    pure_normalized_consensus[
                        subset_mask
                    ],
                    dtype=np.float64,
                )
            ),
            "Median Pure consensus": float(
                np.median(
                    pure_normalized_consensus[
                        subset_mask
                    ]
                )
            ),
            "Mean Full reliability": float(
                np.mean(
                    full_reliability[
                        subset_mask
                    ],
                    dtype=np.float64,
                )
            ),
            "Median Full reliability": float(
                np.median(
                    full_reliability[
                        subset_mask
                    ]
                )
            ),
            "Mean Pure reliability": float(
                np.mean(
                    pure_reliability[
                        subset_mask
                    ],
                    dtype=np.float64,
                )
            ),
            "Median Pure reliability": float(
                np.median(
                    pure_reliability[
                        subset_mask
                    ]
                )
            ),
            "Median Full pre-clipping score": float(
                np.median(
                    full_score[
                        subset_mask
                    ]
                )
            ),
            "Median Pure pre-clipping score": float(
                np.median(
                    pure_score[
                        subset_mask
                    ]
                )
            ),
        }
    )


selection_subset_summary_df = pd.DataFrame(
    subset_rows
)




In [ ]:
# ============================================================
# 22. Validation summary
# ============================================================

validation_df = pd.DataFrame(
    [
        {
            "Check": "Dataset and diagnostic-array lengths",
            "Passed": (
                n_samples
                == pure_score.size
                == full_agreement_count.size
                == pure_agreement_count.size
                == full_uncertainty.size
                == pure_uncertainty.size
                == full_reliability.size
                == pure_reliability.size
                == full_selected.size
                == pure_selected.size
            ),
            "Details": (
                f"{n_samples:,} pixels in every array"
            ),
        },
        {
            "Check": "Full exact top-2% count",
            "Passed": (
                full_selected_count
                == expected_selected_count
            ),
            "Details": (
                f"{full_selected_count:,} selected; "
                f"{expected_selected_count:,} expected"
            ),
        },
        {
            "Check": "Pure exact top-2% count",
            "Passed": (
                pure_selected_count
                == expected_selected_count
            ),
            "Details": (
                f"{pure_selected_count:,} selected; "
                f"{expected_selected_count:,} expected"
            ),
        },
        {
            "Check": "Full agreement within valid range",
            "Passed": (
                int(full_agreement_count.min()) >= 0
                and int(full_agreement_count.max())
                <= FULL_SIGNAL_COUNT
            ),
            "Details": (
                f"Observed "
                f"{int(full_agreement_count.min())} to "
                f"{int(full_agreement_count.max())}; "
                f"theoretical range 0 to "
                f"{FULL_SIGNAL_COUNT}"
            ),
        },
        {
            "Check": "Pure agreement within valid range",
            "Passed": (
                int(pure_agreement_count.min()) >= 0
                and int(pure_agreement_count.max())
                <= PURE_SIGNAL_COUNT
            ),
            "Details": (
                f"Observed "
                f"{int(pure_agreement_count.min())} to "
                f"{int(pure_agreement_count.max())}; "
                f"theoretical range 0 to "
                f"{PURE_SIGNAL_COUNT}"
            ),
        },
        {
            "Check": "Full reliability in [0,1]",
            "Passed": (
                float(full_reliability.min())
                >= -reliability_tolerance
                and float(full_reliability.max())
                <= 1.0 + reliability_tolerance
            ),
            "Details": (
                f"{float(full_reliability.min()):.8f} "
                f"to "
                f"{float(full_reliability.max()):.8f}"
            ),
        },
        {
            "Check": "Pure reliability in [0,1]",
            "Passed": (
                float(pure_reliability.min())
                >= -reliability_tolerance
                and float(pure_reliability.max())
                <= 1.0 + reliability_tolerance
            ),
            "Details": (
                f"{float(pure_reliability.min()):.8f} "
                f"to "
                f"{float(pure_reliability.max()):.8f}"
            ),
        },
        {
            "Check": "Full uncertainty in [0,1]",
            "Passed": (
                float(full_uncertainty.min())
                >= -reliability_tolerance
                and float(full_uncertainty.max())
                <= 1.0 + reliability_tolerance
            ),
            "Details": (
                f"{float(full_uncertainty.min()):.8f} "
                f"to "
                f"{float(full_uncertainty.max()):.8f}"
            ),
        },
        {
            "Check": "Pure uncertainty in [0,1]",
            "Passed": (
                float(pure_uncertainty.min())
                >= -reliability_tolerance
                and float(pure_uncertainty.max())
                <= 1.0 + reliability_tolerance
            ),
            "Details": (
                f"{float(pure_uncertainty.min()):.8f} "
                f"to "
                f"{float(pure_uncertainty.max()):.8f}"
            ),
        },
    ]
)


if not validation_df["Passed"].all():

    failed_checks = validation_df.loc[
        ~validation_df["Passed"],
        [
            "Check",
            "Details",
        ],
    ]

    print("\nFailed validation checks:")
    print(failed_checks)

    raise ValueError(
        "At least one validation check failed. "
        "Do not use the results until it is corrected."
    )


print("\nAll validation checks passed.")

# ============================================================
# 23. Experimental settings
# ============================================================

settings_df = pd.DataFrame(
    {
        "Setting": [
            "Dataset",
            "Seed",
            "Number of pixels",
            "Final anomaly budget",
            "Selected pixels per variant",
            "Full internal signals",
            "Pure internal signals",
            "Full agreement range",
            "Pure agreement range",
            "Consensus normalization",
            "High-consensus definition",
            "Full high-consensus threshold",
            "Pure high-consensus threshold",
            "Final score used",
            "Reliability source",
            "Baseline methods included",
        ],
        "Value": [
            "Original uninjected geospatial dataset",
            42,
            n_samples,
            "Top 2%",
            expected_selected_count,
            (
                "AE, IF-feature, IF-embedding, "
                "LOF-feature, LOF-embedding, "
                "MD-feature, MD-embedding, "
                "spatial, context"
            ),
            "AE, spatial, context",
            "0 to 9",
            "0 to 3",
            (
                "Agreement count divided by the "
                "number of internal signals"
            ),
            (
                "Normalized consensus greater than "
                "or equal to 2/3"
            ),
            FULL_HIGH_CONSENSUS_COUNT,
            PURE_HIGH_CONSENSUS_COUNT,
            (
                "Final fused score before final "
                "clipping and output normalization"
            ),
            (
                "full_pure_reliability_diagnostics.npz"
            ),
            (
                "No; this section analyzes the internal "
                "Full and Pure mechanisms"
            ),
        ],
    }
)


# ============================================================
# 24. Save CSV files
# ============================================================

agreement_level_summary_df.to_csv(
    OUTPUT_DIR
    / "Agreement_Level_Summary.csv",
    index=False,
)

full_agreement_summary_df.to_csv(
    OUTPUT_DIR
    / "Full_Agreement_Level_Summary.csv",
    index=False,
)

pure_agreement_summary_df.to_csv(
    OUTPUT_DIR
    / "Pure_Agreement_Level_Summary.csv",
    index=False,
)

variant_summary_df.to_csv(
    OUTPUT_DIR
    / "Pure_Full_Variant_Summary.csv",
    index=False,
)

selection_overlap_df.to_csv(
    OUTPUT_DIR
    / "Pure_Full_Selection_Overlap.csv",
    index=False,
)

selection_subset_summary_df.to_csv(
    OUTPUT_DIR
    / "Pure_Full_Selection_Subsets.csv",
    index=False,
)

trend_summary_df.to_csv(
    OUTPUT_DIR
    / "Agreement_Trend_Statistics.csv",
    index=False,
)





In [ ]:
# ============================================================
# 25. Prepare compact paper-oriented tables
# ============================================================

full_agreement_paper_df = (
    full_agreement_summary_df[
        [
            "Agreement count",
            "Normalized consensus",
            "Pixel count",
            "Dataset share (%)",
            "Median score",
            "Score Q1",
            "Score Q3",
            "Median reliability",
            "Selection rate within level (%)",
            "Share of final selection (%)",
        ]
    ].copy()
)


pure_agreement_paper_df = (
    pure_agreement_summary_df[
        [
            "Agreement count",
            "Normalized consensus",
            "Pixel count",
            "Dataset share (%)",
            "Median score",
            "Score Q1",
            "Score Q3",
            "Median reliability",
            "Selection rate within level (%)",
            "Share of final selection (%)",
        ]
    ].copy()
)


variant_paper_df = variant_summary_df[
    [
        "Variant",
        "Internal signals",
        "Mean consensus, all pixels",
        "Mean consensus, selected pixels",
        "Median consensus, selected pixels",
        "Mean reliability, all pixels",
        "Mean reliability, selected pixels",
        "Median reliability, selected pixels",
        "High-consensus share, selected pixels (%)",
        "Zero-agreement share, selected pixels (%)",
    ]
].copy()


selection_overlap_paper_df = (
    selection_overlap_df[
        [
            "Selected by both",
            "Full-only selected",
            "Pure-only selected",
            "Jaccard similarity",
            "Shared share of Full selection (%)",
            "Shared share of Pure selection (%)",
        ]
    ].copy()
)


# ============================================================
# 26. Save LaTeX table files
# ============================================================

with open(
    OUTPUT_DIR
    / "table_full_agreement_levels.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        full_agreement_paper_df.to_latex(
            index=False,
            float_format="%.4f",
            escape=True,
        )
    )


with open(
    OUTPUT_DIR
    / "table_pure_agreement_levels.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        pure_agreement_paper_df.to_latex(
            index=False,
            float_format="%.4f",
            escape=True,
        )
    )


with open(
    OUTPUT_DIR
    / "table_pure_full_variant_summary.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        variant_paper_df.to_latex(
            index=False,
            float_format="%.4f",
            escape=True,
        )
    )


with open(
    OUTPUT_DIR
    / "table_pure_full_selection_overlap.tex",
    "w",
    encoding="utf-8",
) as latex_file:

    latex_file.write(
        selection_overlap_paper_df.to_latex(
            index=False,
            float_format="%.4f",
            escape=True,
        )
    )






In [ ]:
# ============================================================
# 27. Figure: Full agreement versus final score
# ============================================================

full_nonempty = full_agreement_summary_df[
    full_agreement_summary_df[
        "Pixel count"
    ] > 0
].copy()


full_lower_error = (
    full_nonempty[
        "Median score"
    ]
    - full_nonempty[
        "Score Q1"
    ]
)

full_upper_error = (
    full_nonempty[
        "Score Q3"
    ]
    - full_nonempty[
        "Median score"
    ]
)


plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    full_nonempty[
        "Agreement count"
    ],
    full_nonempty[
        "Median score"
    ],
    yerr=np.vstack(
        [
            full_lower_error,
            full_upper_error,
        ]
    ),
    marker="o",
    capsize=4,
)

plt.xlabel(
    "Full internal agreement count (out of 9)"
)

plt.ylabel(
    "Median pre-final-clipping fused score"
)

plt.title(
    "Full agreement versus final anomaly score"
)

plt.xticks(
    range(
        FULL_SIGNAL_COUNT + 1
    )
)

plt.grid(
    axis="y",
    alpha=0.3,
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Full_Agreement_vs_Median_Score.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()



In [ ]:
# ============================================================
# 28. Figure: Pure agreement versus final score
# ============================================================

pure_nonempty = pure_agreement_summary_df[
    pure_agreement_summary_df[
        "Pixel count"
    ] > 0
].copy()


pure_lower_error = (
    pure_nonempty[
        "Median score"
    ]
    - pure_nonempty[
        "Score Q1"
    ]
)

pure_upper_error = (
    pure_nonempty[
        "Score Q3"
    ]
    - pure_nonempty[
        "Median score"
    ]
)


plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    pure_nonempty[
        "Agreement count"
    ],
    pure_nonempty[
        "Median score"
    ],
    yerr=np.vstack(
        [
            pure_lower_error,
            pure_upper_error,
        ]
    ),
    marker="o",
    capsize=4,
)

plt.xlabel(
    "Pure internal agreement count (out of 3)"
)

plt.ylabel(
    "Median pre-final-clipping fused score"
)

plt.title(
    "Pure agreement versus final anomaly score"
)

plt.xticks(
    range(
        PURE_SIGNAL_COUNT + 1
    )
)

plt.grid(
    axis="y",
    alpha=0.3,
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Pure_Agreement_vs_Median_Score.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()



In [ ]:
# ============================================================
# 29. Figure: consensus versus final-selection rate
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    full_nonempty[
        "Normalized consensus"
    ],
    full_nonempty[
        "Selection rate within level (%)"
    ],
    marker="o",
    label="Full",
)

plt.plot(
    pure_nonempty[
        "Normalized consensus"
    ],
    pure_nonempty[
        "Selection rate within level (%)"
    ],
    marker="s",
    label="Pure",
)

plt.xlabel(
    "Normalized internal anomaly agreement"
)

plt.ylabel(
    "Final top-2% selection rate within agreement level (%)"
)

plt.title(
    "Internal anomaly agreement and final anomaly selection"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3,
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Consensus_vs_Final_Selection_Rate.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()




In [ ]:
# ============================================================
# 30. Figure: Full-Pure selected-set subsets
# ============================================================

selected_subset_plot_df = (
    selection_subset_summary_df[
        selection_subset_summary_df[
            "Selection subset"
        ].isin(
            [
                "Both selected",
                "Full only",
                "Pure only",
            ]
        )
    ].copy()
)


plt.figure(
    figsize=(8, 5)
)

plt.bar(
    selected_subset_plot_df[
        "Selection subset"
    ],
    selected_subset_plot_df[
        "Pixel count"
    ],
)

plt.xlabel(
    "Full-Pure selection category"
)

plt.ylabel(
    "Number of pixels"
)

plt.title(
    "Shared and variant-specific top-2% detections"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "Pure_Full_Selection_Subset_Counts.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()



In [ ]:
# ============================================================
# 31. Paper-use guidance
# ============================================================

paper_use_guide_df = pd.DataFrame(
    {
        "Result": [
            "Agreement versus score",
            "Agreement versus reliability",
            "Selection rate by agreement",
            "Normalized consensus",
            "Full-Pure reliability comparison",
            "Full-Pure selected-set overlap",
            "Causal interpretation",
        ],
        "Interpretation": [
            (
                "Shows whether larger internal agreement is "
                "associated with larger final fused scores."
            ),
            (
                "This is a mechanism-consistency check rather "
                "than independent validation because agreement "
                "contributes to the reliability mechanism."
            ),
            (
                "Shows the proportion of each agreement level "
                "that enters the final exact top-2% selection."
            ),
            (
                "Agreement count divided by the number of "
                "internal signals. Normalization is required "
                "because Full has nine signals and Pure has "
                "three."
            ),
            (
                "Full and Pure reliability values are bounded "
                "and can be compared descriptively. Their raw "
                "pre-final-clipping scores should not be "
                "directly compared as if they share an "
                "identical numerical scale."
            ),
            (
                "Shows shared and variant-specific detections "
                "and therefore indicates complementarity, not "
                "independent detection accuracy."
            ),
            (
                "Observed relationships characterize the "
                "operation of the proposed mechanism and must "
                "not be interpreted as causal evidence."
            ),
        ],
    }
)

# ============================================================
# 32. Save the final Excel workbook
# ============================================================

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl",
) as writer:

    validation_df.to_excel(
        writer,
        sheet_name="Validation",
        index=False,
    )

    agreement_audit_df.to_excel(
        writer,
        sheet_name="Agreement Audit",
        index=False,
    )

    mask_audit_df.to_excel(
        writer,
        sheet_name="Mask Audit",
        index=False,
    )

    agreement_level_summary_df.to_excel(
        writer,
        sheet_name="Agreement Level Summary",
        index=False,
    )

    full_agreement_summary_df.to_excel(
        writer,
        sheet_name="Full Agreement Levels",
        index=False,
    )

    pure_agreement_summary_df.to_excel(
        writer,
        sheet_name="Pure Agreement Levels",
        index=False,
    )

    trend_summary_df.to_excel(
        writer,
        sheet_name="Agreement Trends",
        index=False,
    )

    variant_summary_df.to_excel(
        writer,
        sheet_name="Pure Full Summary",
        index=False,
    )

    selection_overlap_df.to_excel(
        writer,
        sheet_name="Selection Overlap",
        index=False,
    )

    selection_subset_summary_df.to_excel(
        writer,
        sheet_name="Selection Subsets",
        index=False,
    )

    settings_df.to_excel(
        writer,
        sheet_name="Experimental Settings",
        index=False,
    )

    paper_use_guide_df.to_excel(
        writer,
        sheet_name="Paper Use Guide",
        index=False,
    )

    workbook = writer.book

    for worksheet in workbook.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = (
            worksheet.dimensions
        )

        for header_cell in worksheet[1]:

            header_cell.font = Font(
                bold=True
            )

            header_cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        for column_cells in worksheet.columns:

            column_letter = (
                column_cells[0].column_letter
            )

            maximum_length = max(
                len(
                    str(
                        cell.value
                    )
                )
                if cell.value is not None
                else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[
                column_letter
            ].width = min(
                maximum_length + 2,
                48,
            )


# ============================================================
# 33. Completion report
# ============================================================

print("\n" + "=" * 76)
print("RELIABILITY AND AGREEMENT ANALYSIS COMPLETED")
print("=" * 76)

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nExcel workbook:")
print(OUTPUT_EXCEL)

print("\nSaved agreement arrays:")
print(FULL_AGREEMENT_PATH)
print(PURE_AGREEMENT_PATH)

print("\nSaved normalized consensus arrays:")
print(FULL_CONSENSUS_PATH)
print(PURE_CONSENSUS_PATH)

print("\nSaved selection subset labels:")
print(SELECTION_SUBSET_PATH)

print("\nPrimary result sheets:")
print("1. Full Agreement Levels")
print("2. Pure Agreement Levels")
print("3. Agreement Trends")
print("4. Pure Full Summary")
print("5. Selection Overlap")
print("6. Selection Subsets")

print("\nPrimary figures:")
print("1. Full_Agreement_vs_Median_Score.png")
print("2. Pure_Agreement_vs_Median_Score.png")
print("3. Consensus_vs_Final_Selection_Rate.png")
print("4. Pure_Full_Selection_Subset_Counts.png")

print("\nAll validation checks passed.")